# Homework 05: Data Storage

This notebook builds a reproducible storage layer with environment-driven paths, CSV and Parquet files, reload validation, and reusable IO functions.

In [1]:
# Packages used: numpy, pandas, pyarrow, python-dotenv

## 1. Environment-Driven Paths

`.env` keeps folder settings outside the code. Both paths remain relative to this homework folder.

In [2]:
import datetime as dt
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('.env'))
RAW = Path(os.getenv('DATA_DIR_RAW', 'data/raw'))
PROCESSED = Path(os.getenv('DATA_DIR_PROCESSED', 'data/processed'))
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print('Raw directory:', RAW.resolve())
print('Processed directory:', PROCESSED.resolve())

Raw directory: D:\Desktop\NYU\FRE Bootcamp\Part3 FRE Bootcamp IV\bootcamp_project\homework\homework05\data\raw
Processed directory: D:\Desktop\NYU\FRE Bootcamp\Part3 FRE Bootcamp IV\bootcamp_project\homework\homework05\data\processed


## 2. Create the Sample Data

The fixed seed makes the same 20-row price series every time.

In [3]:
rng = np.random.default_rng(5)
df = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=20, freq='D'),
    'ticker': ['SPY'] * 20,
    'price': 470 + rng.normal(0, 2, 20).cumsum(),
    'volume': rng.integers(50_000_000, 90_000_000, 20),
})
df.head()

,date,ticker,price,volume
0,2024-01-01,SPY,468.396137,88363975
1,2024-01-02,SPY,465.747419,72223844
2,2024-01-03,SPY,465.250696,86140472
3,2024-01-04,SPY,466.091586,60858064
4,2024-01-05,SPY,468.363679,64460729


## 3. Save CSV and Parquet

One timestamp is reused so the two files represent the same data snapshot.

In [4]:
stamp = dt.datetime.now().strftime('%Y%m%d-%H%M%S')
csv_path = RAW / f'etf_prices_{stamp}.csv'
parquet_path = PROCESSED / f'etf_prices_{stamp}.parquet'

df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path, index=False)
print('Saved:', csv_path)
print('Saved:', parquet_path)

Saved: data\raw\etf_prices_20260824-103448.csv
Saved: data\processed\etf_prices_20260824-103448.parquet


## 4. Reload and Validate

The validation checks shape and the critical date, price, and volume types.

In [5]:
def validate_loaded(original: pd.DataFrame, reloaded: pd.DataFrame) -> dict[str, bool]:
    """Compare shape and critical data types after a storage round trip."""
    return {
        'shape_equal': original.shape == reloaded.shape,
        'date_is_datetime': pd.api.types.is_datetime64_any_dtype(reloaded['date']),
        'price_is_numeric': pd.api.types.is_numeric_dtype(reloaded['price']),
        'volume_is_integer': pd.api.types.is_integer_dtype(reloaded['volume']),
    }

csv_loaded = pd.read_csv(csv_path, parse_dates=['date'])
parquet_loaded = pd.read_parquet(parquet_path)
validation = pd.DataFrame({
    'csv': validate_loaded(df, csv_loaded),
    'parquet': validate_loaded(df, parquet_loaded),
})
validation.loc['all_checks_pass'] = validation.all(axis=0)
validation

,csv,parquet
shape_equal,True,True
date_is_datetime,True,True
price_is_numeric,True,True
volume_is_integer,True,True
all_checks_pass,True,True


## 5. Reusable IO Utilities

The file suffix chooses the storage method. Parent folders are created automatically, and Parquet errors include an installation hint.

In [6]:
def detect_format(path: str | Path) -> str:
    """Return the supported storage format identified by a file suffix."""
    suffix = Path(path).suffix.lower()
    if suffix == '.csv':
        return 'csv'
    if suffix in {'.parquet', '.pq', '.parq'}:
        return 'parquet'
    raise ValueError(f'Unsupported format: {suffix}')


def write_df(frame: pd.DataFrame, path: str | Path) -> Path:
    """Write a DataFrame as CSV or Parquet based on its target suffix."""
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    if detect_format(target) == 'csv':
        frame.to_csv(target, index=False)
    else:
        try:
            frame.to_parquet(target, index=False)
        except ImportError as exc:
            raise RuntimeError('Install pyarrow or fastparquet for Parquet support.') from exc
    return target


def read_df(path: str | Path) -> pd.DataFrame:
    """Read a CSV or Parquet file based on its suffix."""
    source = Path(path)
    if detect_format(source) == 'csv':
        header = pd.read_csv(source, nrows=0).columns
        return pd.read_csv(source, parse_dates=['date'] if 'date' in header else None)
    try:
        return pd.read_parquet(source)
    except ImportError as exc:
        raise RuntimeError('Install pyarrow or fastparquet for Parquet support.') from exc

In [7]:
utility_csv = write_df(df, RAW / 'utility_demo.csv')
utility_parquet = write_df(df, PROCESSED / 'utility_demo.parquet')
utility_checks = pd.DataFrame({
    'csv': validate_loaded(df, read_df(utility_csv)),
    'parquet': validate_loaded(df, read_df(utility_parquet)),
})
utility_checks

,csv,parquet
shape_equal,True,True
date_is_datetime,True,True
price_is_numeric,True,True
volume_is_integer,True,True


## 6. Storage Assumptions

CSV is the portable raw representation but requires date parsing on reload. Parquet is the processed representation because it preserves data types and supports efficient analytical reads. The validation table confirms both round trips preserve the expected structure.